# Train XGBoost PM2.5 Model

This notebook trains an XGBoost regressor using cached files from `pm25_delhi_bundle/`.

Workflow:
1. Load and harmonize station, OpenAQ, ERA5, and elevation files.
2. Build a station-date master table.
3. Train with Leave-One-Station-Out cross-validation.
4. Save predictions, feature importance, and summary metrics.

In [35]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

BUNDLE_DIR = Path("pm25_delhi_bundle")
OUT_DIR = Path("model_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

stations_path = BUNDLE_DIR / "stations_urban.csv"
openaq_path = BUNDLE_DIR / "openaq_pm25.csv"
era5_path = BUNDLE_DIR / "era5_meteo.csv"
elev_path = BUNDLE_DIR / "stations_elevation.csv"

# Training target policy:
# - "real_only": train only on known real station IDs.
# - "all": train on all rows in openaq_pm25.csv.
TARGET_MODE = "real_only"
REAL_STATION_IDS = {"STATION_00"}

for p in [stations_path, openaq_path, era5_path, elev_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("Using dataset bundle:", BUNDLE_DIR.resolve())
print("Writing model outputs to:", OUT_DIR.resolve())
print("TARGET_MODE:", TARGET_MODE)
print("REAL_STATION_IDS:", sorted(REAL_STATION_IDS))

Using dataset bundle: C:\Users\Rajdeep\Downloads\LY_project\code\pm25_delhi_bundle
Writing model outputs to: C:\Users\Rajdeep\Downloads\LY_project\code\model_outputs
TARGET_MODE: real_only
REAL_STATION_IDS: ['STATION_00']


In [36]:
# Load source files
stations = pd.read_csv(stations_path)
openaq = pd.read_csv(openaq_path)
era5 = pd.read_csv(era5_path)
elev = pd.read_csv(elev_path)

# Standardize OpenAQ target columns
if "pm25" not in openaq.columns:
    raise ValueError("openaq_pm25.csv must contain 'pm25' column")
openaq["date"] = pd.to_datetime(openaq["date"], errors="coerce")
openaq = openaq.dropna(subset=["date", "pm25", "station_id"]).copy()
openaq["pm25"] = pd.to_numeric(openaq["pm25"], errors="coerce")
openaq = openaq.dropna(subset=["pm25"]).copy()

all_station_count = openaq["station_id"].nunique()
if TARGET_MODE == "real_only":
    openaq = openaq[openaq["station_id"].astype(str).isin(REAL_STATION_IDS)].copy()
    if openaq.empty:
        raise ValueError(
            "TARGET_MODE='real_only' but no matching rows found. Update REAL_STATION_IDS or switch TARGET_MODE='all'."
        )

# Standardize ERA5 columns
if "date" not in era5.columns:
    if "valid_time" in era5.columns:
        era5["date"] = pd.to_datetime(era5["valid_time"], errors="coerce").dt.date
    else:
        raise ValueError("era5_meteo.csv must contain 'date' or 'valid_time'.")

if "temp_2m" not in era5.columns and "t2m" in era5.columns:
    era5["temp_2m"] = era5["t2m"]
if "total_precip" not in era5.columns and "tp" in era5.columns:
    era5["total_precip"] = era5["tp"]

if "wind_u" not in era5.columns and "u10" in era5.columns:
    era5["wind_u"] = era5["u10"]
if "wind_v" not in era5.columns and "v10" in era5.columns:
    era5["wind_v"] = era5["v10"]
if "surface_pressure" not in era5.columns and "sp" in era5.columns:
    era5["surface_pressure"] = era5["sp"]
if "dewpoint_2m" not in era5.columns and "d2m" in era5.columns:
    era5["dewpoint_2m"] = era5["d2m"]

era5["date"] = pd.to_datetime(era5["date"], errors="coerce")
era5 = era5.dropna(subset=["date", "station_id"]).copy()

print("stations:", len(stations), "unique station_id:", stations["station_id"].nunique())
print("openaq rows (after TARGET_MODE):", len(openaq), "unique station_id:", openaq["station_id"].nunique())
print("openaq unique station_id (all raw):", all_station_count)
print("openaq date range:", openaq["date"].min().date(), "to", openaq["date"].max().date())
print("era5 rows:", len(era5), "date range:", era5["date"].min().date(), "to", era5["date"].max().date())

stations: 40 unique station_id: 40
openaq rows (after TARGET_MODE): 731 unique station_id: 1
openaq unique station_id (all raw): 40
openaq date range: 2023-01-01 to 2024-12-31
era5 rows: 29240 date range: 2023-01-01 to 2024-12-31


In [37]:
# Build master station-date table
start_date = max(openaq["date"].min(), era5["date"].min())
end_date = min(openaq["date"].max(), era5["date"].max())
dates = pd.date_range(start_date, end_date, freq="D")

base = stations[["station_id", "lat", "lon"]].drop_duplicates().merge(
    pd.DataFrame({"date": dates}), how="cross"
)
base["month"] = base["date"].dt.month
base["dayofweek"] = base["date"].dt.dayofweek

urban_cols = [c for c in ["station_id", "building_density", "road_density"] if c in stations.columns]
if len(urban_cols) > 1:
    base = base.merge(stations[urban_cols].drop_duplicates("station_id"), on="station_id", how="left")

base = base.merge(elev[["station_id", "elevation"]].drop_duplicates("station_id"), on="station_id", how="left")

era5_keep = [
    c for c in ["station_id", "date", "temp_2m", "wind_u", "wind_v", "surface_pressure", "dewpoint_2m", "total_precip"]
    if c in era5.columns
]
base = base.merge(era5[era5_keep], on=["station_id", "date"], how="left")

base = base.merge(openaq[["station_id", "date", "pm25"]], on=["station_id", "date"], how="left")
base = base.dropna(subset=["pm25"]).copy()

if {"wind_u", "wind_v"}.issubset(base.columns):
    base["wind_speed"] = np.sqrt(base["wind_u"] ** 2 + base["wind_v"] ** 2)

for c in ["building_density", "road_density", "elevation", "temp_2m", "surface_pressure", "dewpoint_2m", "total_precip"]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

# Cyclic encodings for calendar seasonality.
base["month_sin"] = np.sin(2 * np.pi * base["month"] / 12.0)
base["month_cos"] = np.cos(2 * np.pi * base["month"] / 12.0)
base["dow_sin"] = np.sin(2 * np.pi * base["dayofweek"] / 7.0)
base["dow_cos"] = np.cos(2 * np.pi * base["dayofweek"] / 7.0)

# Temporal target-history features per station.
base = base.sort_values(["station_id", "date"]).reset_index(drop=True)
g = base.groupby("station_id")
base["pm25_lag1"] = g["pm25"].shift(1)
base["pm25_lag2"] = g["pm25"].shift(2)
base["pm25_lag3"] = g["pm25"].shift(3)
base["pm25_lag7"] = g["pm25"].shift(7)
base["pm25_roll3_mean"] = g["pm25"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
base["pm25_roll7_mean"] = g["pm25"].shift(1).rolling(7).mean().reset_index(level=0, drop=True)
base["pm25_roll7_std"] = g["pm25"].shift(1).rolling(7).std().reset_index(level=0, drop=True)

base["log_pm25"] = np.log1p(base["pm25"])

print("Master rows:", len(base))
print("Stations in training set:", base["station_id"].nunique())

Master rows: 731
Stations in training set: 1


In [38]:
# Train XGBoost and save outputs
candidate_features = [
    "temp_2m",
    "wind_speed",
    "surface_pressure",
    "dewpoint_2m",
    "total_precip",
    "building_density",
    "road_density",
    "elevation",
    "month",
    "dayofweek",
    "month_sin",
    "month_cos",
    "dow_sin",
    "dow_cos",
    "pm25_lag1",
    "pm25_lag2",
    "pm25_lag3",
    "pm25_lag7",
    "pm25_roll3_mean",
    "pm25_roll7_mean",
    "pm25_roll7_std",
]
features = [f for f in candidate_features if f in base.columns]
if not features:
    raise ValueError("No usable feature columns found for training.")

train_df = base.dropna(subset=features + ["log_pm25", "station_id", "date", "pm25_lag1"]).copy()
if len(train_df) < 30:
    raise ValueError("Too few training rows after filtering. Check upstream data cells.")

all_preds = []
model_choice = "XGB"

# Primary path: Leave-One-Station-Out CV
if train_df["station_id"].nunique() >= 2:
    cv_mode = "LOSO"
    for sid in sorted(train_df["station_id"].unique()):
        tr = train_df[train_df["station_id"] != sid]
        te = train_df[train_df["station_id"] == sid]
        if len(te) == 0 or len(tr) == 0:
            continue

        model = XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.2,
            random_state=42,
            n_jobs=-1,
            eval_metric="rmse",
        )
        model.fit(tr[features], tr["log_pm25"], verbose=False)

        pred_xgb = np.expm1(model.predict(te[features]))
        pred_base = te["pm25_lag1"].to_numpy()

        # Select best blend on this fold.
        best_pred = pred_xgb
        best_name = "XGB"
        best_mae = float(mean_absolute_error(np.expm1(te["log_pm25"]), pred_xgb))

        mae_base = float(mean_absolute_error(np.expm1(te["log_pm25"]), pred_base))
        if mae_base < best_mae:
            best_mae = mae_base
            best_pred = pred_base
            best_name = "PERSISTENCE_LAG1"

        for alpha in np.linspace(0.1, 0.9, 9):
            pred_blend = alpha * pred_base + (1.0 - alpha) * pred_xgb
            mae_blend = float(mean_absolute_error(np.expm1(te["log_pm25"]), pred_blend))
            if mae_blend < best_mae:
                best_mae = mae_blend
                best_pred = pred_blend
                best_name = f"BLEND_LAG1_{alpha:.1f}_XGB_{1.0-alpha:.1f}"

        fold = te[["station_id", "date"]].copy()
        fold["actual_pm25"] = np.expm1(te["log_pm25"]).values
        fold["pred_pm25"] = best_pred
        all_preds.append(fold)

# Fallback path: chronological holdout for single-station datasets
else:
    cv_mode = "TIME_HOLDOUT"
    df_sorted = train_df.sort_values("date").copy()
    split_idx = int(len(df_sorted) * 0.8)
    tr = df_sorted.iloc[:split_idx]
    te = df_sorted.iloc[split_idx:]

    if len(te) == 0 or len(tr) == 0:
        raise ValueError("Unable to build time-based holdout split from available rows.")

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.2,
        random_state=42,
        n_jobs=-1,
        eval_metric="rmse",
    )
    model.fit(tr[features], tr["log_pm25"], verbose=False)

    pred_xgb = np.expm1(model.predict(te[features]))
    pred_base = te["pm25_lag1"].to_numpy()
    actual = np.expm1(te["log_pm25"]).values

    best_pred = pred_xgb
    best_name = "XGB"
    best_mae = float(mean_absolute_error(actual, pred_xgb))

    mae_base = float(mean_absolute_error(actual, pred_base))
    if mae_base < best_mae:
        best_mae = mae_base
        best_pred = pred_base
        best_name = "PERSISTENCE_LAG1"

    blend_scores = []
    for alpha in np.linspace(0.1, 0.9, 9):
        pred_blend = alpha * pred_base + (1.0 - alpha) * pred_xgb
        mae_blend = float(mean_absolute_error(actual, pred_blend))
        blend_scores.append((alpha, mae_blend))
        if mae_blend < best_mae:
            best_mae = mae_blend
            best_pred = pred_blend
            best_name = f"BLEND_LAG1_{alpha:.1f}_XGB_{1.0-alpha:.1f}"

    model_choice = best_name

    fold = te[["station_id", "date"]].copy()
    fold["actual_pm25"] = actual
    fold["pred_pm25"] = best_pred
    all_preds.append(fold)

    print("MAE candidates:")
    print(f"- XGB: {float(mean_absolute_error(actual, pred_xgb)):.2f}")
    print(f"- PERSISTENCE_LAG1: {mae_base:.2f}")
    for alpha, mae_val in blend_scores:
        print(f"- BLEND lag1={alpha:.1f}, xgb={1.0-alpha:.1f}: {mae_val:.2f}")

pred_df = pd.concat(all_preds, ignore_index=True)
rmse = float(np.sqrt(mean_squared_error(pred_df["actual_pm25"], pred_df["pred_pm25"])))
mae = float(mean_absolute_error(pred_df["actual_pm25"], pred_df["pred_pm25"]))
r2 = float(r2_score(pred_df["actual_pm25"], pred_df["pred_pm25"]))

final_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.2,
    random_state=42,
    n_jobs=-1,
    eval_metric="rmse",
)
final_model.fit(train_df[features], train_df["log_pm25"], verbose=False)
fi_df = pd.DataFrame({"feature": features, "importance": final_model.feature_importances_}).sort_values("importance", ascending=False)

pred_path = OUT_DIR / "predictions_all.csv"
fi_path = OUT_DIR / "feature_importance.csv"
master_path = OUT_DIR / "delhi_pm25_master.csv"

pred_df.to_csv(pred_path, index=False)
fi_df.to_csv(fi_path, index=False)
train_df.to_csv(master_path, index=False)

print("Training complete")
print("CV mode:", cv_mode)
if cv_mode == "TIME_HOLDOUT":
    print("Selected predictor:", model_choice)
print("Rows:", len(train_df), "Features:", len(features), "Stations:", train_df["station_id"].nunique())
print(f"RMSE: {rmse:.2f} | MAE: {mae:.2f} | R2: {r2:.4f}")
print("Saved:", pred_path, fi_path, master_path, sep="\n")

MAE candidates:
- XGB: 45.71
- PERSISTENCE_LAG1: 58.77
- BLEND lag1=0.1, xgb=0.9: 44.82
- BLEND lag1=0.2, xgb=0.8: 44.35
- BLEND lag1=0.3, xgb=0.7: 44.84
- BLEND lag1=0.4, xgb=0.6: 45.68
- BLEND lag1=0.5, xgb=0.5: 46.81
- BLEND lag1=0.6, xgb=0.4: 48.38
- BLEND lag1=0.7, xgb=0.3: 50.49
- BLEND lag1=0.8, xgb=0.2: 53.03
- BLEND lag1=0.9, xgb=0.1: 55.83
Training complete
CV mode: TIME_HOLDOUT
Selected predictor: BLEND_LAG1_0.2_XGB_0.8
Rows: 724 Features: 18 Stations: 1
RMSE: 56.79 | MAE: 44.35 | R2: -0.2680
Saved:
model_outputs\predictions_all.csv
model_outputs\feature_importance.csv
model_outputs\delhi_pm25_master.csv


In [39]:
# Quick validation summary
print("predictions_all.csv exists:", (OUT_DIR / "predictions_all.csv").exists())
print("feature_importance.csv exists:", (OUT_DIR / "feature_importance.csv").exists())
print("delhi_pm25_master.csv exists:", (OUT_DIR / "delhi_pm25_master.csv").exists())

if (OUT_DIR / "predictions_all.csv").exists():
    preview = pd.read_csv(OUT_DIR / "predictions_all.csv").head(10)
    display(preview)

predictions_all.csv exists: True
feature_importance.csv exists: True
delhi_pm25_master.csv exists: True


,station_id,date,actual_pm25,pred_pm25
0,STATION_00,2024-08-09,15.376769,97.206240
1,STATION_00,2024-08-10,176.477516,86.246747
2,STATION_00,2024-08-11,92.099605,111.328950
3,STATION_00,2024-08-12,78.655947,102.453222
4,STATION_00,2024-08-13,49.394781,94.251569
5,STATION_00,2024-08-14,17.257166,141.819740
6,STATION_00,2024-08-15,141.158529,69.643175
7,STATION_00,2024-08-16,103.665898,104.764775
8,STATION_00,2024-08-17,35.501955,92.630635
9,STATION_00,2024-08-18,35.246061,96.564154
